## settings

In [3]:
!git clone https://github.com/hiyouga/LLaMA-Factory.git


Cloning into 'LLaMA-Factory'...


In [1]:
import sys
print(sys.executable)


c:\Users\mohammed\AppData\Local\Programs\Python\Python313\python.exe


In [3]:
import wandb
from dotenv import load_dotenv
import os

load_dotenv()

wandb.login()

hf_token = os.getenv('HF_API_KEY')
!hf auth login --token {hf_token}

wandb: Currently logged in as: mahammadtaha74 (mahammadtaha74-al-azhar-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `hf`CLI if you want to set the git credential as well.
Token is valid (permission: fineGrained).
The token `llm_finetune` has been saved to E:\tools\hf_models\stored_tokens
Your token has been saved to E:\tools\hf_models\token
Login successful.
The current active token is: `llm_finetune`


## Imports

In [4]:
import json
import os
from os.path import join
import random
from tqdm.auto import tqdm
import requests

from pydantic import BaseModel, Field
from typing import List, Optional, Literal
from datetime import datetime

import json_repair

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

from dotenv import load_dotenv
load_dotenv()

data_dir = os.getenv("data_dir")
base_model_id = os.getenv("base_model_id")

device = os.getenv("device")
torch_dtype = None

def parse_json(text):
    try:
        return json_repair.loads(text)
    except:
        return None

c:\Users\mohammed\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Format Finetuning Datasets

In [10]:
sft_data_path = os.path.join(data_dir, "fine_tune_dataset","sft.jsonl")
llm_finetuning_data = []

system_prompt = "/n".join([
    "you are a professional NLP data parser",
    "follow the provided `task` by the user and the `output_schema` to generate `OutPut JSON`",
    "dont generate any introduction or conclusion"
])

for line in open(sft_data_path, "r"):
    if line.strip() == "":
        continue
    rec = json.loads(line.strip())

    llm_finetuning_data.append({
        "system": system_prompt,
        "instruction": "\n".join([
            "# Story:",
            rec["story"],

            "# Task:",
            rec["task"],

            "# Output Scheme:",
            rec["output_scheme"],
            "",

            "# Output JSON:",
            "```json"

        ]),
        "input": "",
        "output": "\n".join([
            "```json",
            json.dumps(rec["response"], ensure_ascii=False, default=str),
            "```"
        ]),
        "history": []
    })

    random.Random(101).shuffle(llm_finetuning_data)

In [11]:
len(llm_finetuning_data)

2766

In [ ]:
train_sample_sz = 2700

train_ds = llm_finetuning_data[:train_sample_sz]
eval_ds = llm_finetuning_data[train_sample_sz:]

os.makedirs(join(data_dir, "fine_tune_dataset", "llamafactory-finetune-data"), exist_ok=True)

with open(join(data_dir, "fine_tune_dataset", "llamafactory-finetune-data", "train.json"), "w") as dest:
    json.dump(train_ds, dest, ensure_ascii=False, default=str)

with open(join(data_dir, "fine_tune_dataset", "llamafactory-finetune-data", "val.json"), "w", encoding="utf8") as dest:
    json.dump(eval_ds, dest, ensure_ascii=False, default=str)

In [12]:
# Path to your dataset_info.json
file_path = "/content/LLaMA-Factory/data/dataset_info.json"

# New entries to append
new_entries = {
    "news_finetune_train": {
        "file_name": "/content/Fine-Tune/app/assets/fine_tune_dataset/llamaFactory finetune data/train.json",
        "columns": {
            "prompt": "instruction",
            "query": "input",
            "response": "output",
            "system": "system",
            "history": "history"
        }
    },
    "news_finetune_val": {
        "file_name": "/content/Fine-Tune/app/assets/fine_tune_dataset/llamaFactory finetune data/val.json",
        "columns": {
            "prompt": "instruction",
            "query": "input",
            "response": "output",
            "system": "system",
            "history": "history"
        }
    }
}

# Load existing JSON
with open(file_path, "r") as f:
    data = json.load(f)

# Append new entries (overwrite if keys already exist)
data.update(new_entries)

# Save back to file
with open(file_path, "w") as f:
    json.dump(data, f, indent=4)

print("✅ dataset_info.json updated successfully!")

✅ dataset_info.json updated successfully!


## fine tune configrations

In [4]:
%%writefile /content/LLaMA-Factory/examples/train_lora/news_finetune.yaml

### model
model_name_or_path: Qwen/Qwen2.5-1.5B-Instruct
trust_remote_code: true

### method
stage: sft
do_train: true
finetuning_type: lora
lora_rank: 64
lora_target: all

### dataset
dataset: news_finetune_train
dataset_dir: /content/LLaMA-Factory/data
eval_dataset: news_finetune_val
template: qwen
cutoff_len: 3500
# max_samples: 50
overwrite_cache: true
preprocessing_num_workers: 16

### output
# resume_from_checkpoint: /content/Fine-Tune/app/my_model/checkpoint-1500
output_dir: /content/Fine-Tune/app/my_model
logging_steps: 10
save_steps: 500
plot_loss: true
# overwrite_output_dir: true

### train
per_device_train_batch_size: 1
gradient_accumulation_steps: 4
learning_rate: 5e-5
num_train_epochs: 3.0
lr_scheduler_type: cosine
warmup_ratio: 0.1
bf16: true
ddp_timeout: 180000000

### eval
# val_size: 0.1
per_device_eval_batch_size: 1
eval_strategy: steps
eval_steps: 100

report_to: wandb
run_name: newsx-finetune-llamafactory

push_to_hub: true
export_hub_model_id: "MohammedTaha00/news-analyzer"
hub_private_repo: true
hub_strategy: checkpoint

Writing /content/LLaMA-Factory/examples/train_lora/news_finetune.yaml


In [6]:
from peft import PeftModel

base_model_path = r"E:\tools\hf_models\qwen"

def load_model_and_tokenizer(model_id, cache_dir, use_adapter=False, adapter_path=None):
    try:
        # Load tokenizer
        tokenizer = AutoTokenizer.from_pretrained(
            model_id,
            cache_dir=cache_dir
        )

        # Load base model
        model = AutoModelForCausalLM.from_pretrained(
            model_id,
            cache_dir=cache_dir,
            torch_dtype=torch_dtype
        )

        # Apply fine-tuned adapter if specified
        if use_adapter and adapter_path:
            model = PeftModel.from_pretrained(model, adapter_path)
            print(f"Adapter loaded from {adapter_path}")

        # Move model to GPU
        model = model.to("cuda")
        return model, tokenizer

    except Exception as e:
        print(f"Error loading model or tokenizer: {str(e)}")
        return None, None

In [7]:
import torch
print("Torch built with CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("GPU detected:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")



Torch built with CUDA: 12.4
CUDA available: True
GPU detected: NVIDIA GeForce GTX 1650


# fine tuning


In [ ]:
%cd /content/LLaMA-Factory
%pip install -e .

In [ ]:
%llamafactory-cli train /content/LLaMA-Factory/examples/train_lora/news_finetune.yaml


In [8]:
finetuned_model_id = r"E:\tools\fine tuned adaptor"

print("Loading base model...")
base_model, tokenizer = load_model_and_tokenizer(
    model_id=base_model_id,
    cache_dir=base_model_path
)

# Load fine-tuned model (with adapter)
print("Loading fine-tuned model with adapter...")
finetuned_model, _ = load_model_and_tokenizer(
    model_id=base_model_id,
    cache_dir=base_model_path,
    use_adapter=True,
    adapter_path=finetuned_model_id
)

if base_model is None or finetuned_model is None:
    print("Failed to load one or both models. Exiting.")
else:
    # Save models if they don't already exist
    base_model_save_path = os.path.join(base_model_path, "base_model")
    finetuned_model_save_path = os.path.join(base_model_path, "finetuned_model")
    
        # Save base model
    if os.path.exists(base_model_save_path):
        print(f"Base model already exists at {base_model_save_path}. Skipping save.")
    else:
        try:
            base_model.save_pretrained(base_model_save_path)
            tokenizer.save_pretrained(base_model_save_path)
            print(f"Base model saved to {base_model_save_path}.")
        except Exception as e:
            print(f"Error saving base model: {str(e)}")

    # Save fine-tuned model
    if os.path.exists(finetuned_model_save_path):
        print(f"Fine-tuned model already exists at {finetuned_model_save_path}. Skipping save.")
    else:
        try:
            finetuned_model.save_pretrained(finetuned_model_save_path)
            tokenizer.save_pretrained(finetuned_model_save_path)
            print(f"Fine-tuned model saved to {finetuned_model_save_path}.")
        except Exception as e:
            print(f"Error saving fine-tuned model: {str(e)}")



Loading base model...
Error loading model or tokenizer: CUDA out of memory. Tried to allocate 54.00 MiB. GPU 0 has a total capacity of 4.00 GiB of which 0 bytes is free. Of the allocated memory 10.70 GiB is allocated by PyTorch, and 182.74 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)
Loading fine-tuned model with adapter...
Error loading model or tokenizer: Can't find 'adapter_config.json' at 'E:\tools\fine tuned adaptor'
Failed to load one or both models. Exiting.
